# GridPulse BR — ANEEL Source Profiling

## Objective

Profile the ANEEL Collective Continuity Indicators dataset before designing
the physical Bronze, Silver and Gold layers.

### Questions to answer

- What is the actual source schema?
- What is the dataset grain?
- Which indicators are available?
- Which years and periods are available?
- Are there null values?
- Are there duplicate records?
- Which columns could form a business key?
- How should indicators relate to regulatory limits?
- Are schemas consistent across historical resources?

In [0]:
import requests

SOURCE_URL = (
    "https://dadosabertos.aneel.gov.br/dataset/"
    "d5f0712e-62f6-4736-8dff-9991f10758a7/resource/"
    "d7f70fb1-725c-4748-afeb-65c6a78df550/download/"
    "indicadores-continuidade-coletivos-2020-2029.parquet"
)

response = requests.head(
    SOURCE_URL,
    allow_redirects=True,
    timeout=30
)

print(f"Status code : {response.status_code}")
print(f"Content type: {response.headers.get('content-type')}")
print(f"File size   : {response.headers.get('content-length')}")
print(f"Final URL   : {response.url}")

Status code : 200
Content type: application/octet-stream
File size   : 30076035
Final URL   : https://dadosabertos.aneel.gov.br/dataset/d5f0712e-62f6-4736-8dff-9991f10758a7/resource/d7f70fb1-725c-4748-afeb-65c6a78df550/download/indicadores-continuidade-coletivos-2020-2029.parquet


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gridpulse
COMMENT 'Data objects for the GridPulse BR data engineering project';

CREATE VOLUME IF NOT EXISTS workspace.gridpulse.landing
COMMENT 'Landing area for raw files retrieved from external data sources';

SHOW VOLUMES IN workspace.gridpulse;

database,volume_name
gridpulse,landing


In [0]:
from pathlib import Path
from datetime import datetime, timezone
import requests

# ---------------------------------------------------------
# Source configuration
# ---------------------------------------------------------

SOURCE_URL = (
    "https://dadosabertos.aneel.gov.br/dataset/"
    "d5f0712e-62f6-4736-8dff-9991f10758a7/resource/"
    "d7f70fb1-725c-4748-afeb-65c6a78df550/download/"
    "indicadores-continuidade-coletivos-2020-2029.parquet"
)

SOURCE_NAME = "aneel"
DATASET_NAME = "continuity_indicators"
FILE_NAME = "indicadores-continuidade-coletivos-2020-2029.parquet"

LANDING_PATH = Path(
    f"/Volumes/workspace/gridpulse/landing/"
    f"{SOURCE_NAME}/{DATASET_NAME}"
)

TARGET_FILE = LANDING_PATH / FILE_NAME


# ---------------------------------------------------------
# Create landing directories
# ---------------------------------------------------------

LANDING_PATH.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Download source file
# ---------------------------------------------------------

ingestion_started_at = datetime.now(timezone.utc)

with requests.get(
    SOURCE_URL,
    stream=True,
    timeout=120
) as response:

    response.raise_for_status()

    with open(TARGET_FILE, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

ingestion_finished_at = datetime.now(timezone.utc)


# ---------------------------------------------------------
# Validate landing file
# ---------------------------------------------------------

file_size_bytes = TARGET_FILE.stat().st_size

print("Landing ingestion completed")
print("-" * 60)
print(f"Source        : {SOURCE_NAME}")
print(f"Dataset       : {DATASET_NAME}")
print(f"Target file   : {TARGET_FILE}")
print(f"Size          : {file_size_bytes:,} bytes")
print(f"Started UTC   : {ingestion_started_at.isoformat()}")
print(f"Finished UTC  : {ingestion_finished_at.isoformat()}")

Landing ingestion completed
------------------------------------------------------------
Source        : aneel
Dataset       : continuity_indicators
Target file   : /Volumes/workspace/gridpulse/landing/aneel/continuity_indicators/indicadores-continuidade-coletivos-2020-2029.parquet
Size          : 30,076,035 bytes
Started UTC   : 2026-09-04T13:05:04.413631+00:00
Finished UTC  : 2026-09-04T13:06:02.527929+00:00


In [0]:
files = dbutils.fs.ls(
    "/Volumes/workspace/gridpulse/landing/aneel/continuity_indicators/"
)

display(files)

path,name,size,modificationTime
dbfs:/Volumes/workspace/gridpulse/landing/aneel/continuity_indicators/indicadores-continuidade-coletivos-2020-2029.parquet,indicadores-continuidade-coletivos-2020-2029.parquet,30076035,1788527137000


In [0]:
df_raw = spark.read.parquet(str(TARGET_FILE))

print(f"Rows: {df_raw.count():,}")

df_raw.printSchema()

Rows: 5,042,862
root
 |-- DatGeracaoConjuntoDados: date (nullable = true)
 |-- IdeConjUndConsumidoras: long (nullable = true)
 |-- DscConjUndConsumidoras: string (nullable = true)
 |-- SigAgente: string (nullable = true)
 |-- NumCNPJ: long (nullable = true)
 |-- SigIndicador: string (nullable = true)
 |-- AnoIndice: long (nullable = true)
 |-- NumPeriodoIndice: long (nullable = true)
 |-- VlrIndiceEnviado: double (nullable = true)



In [0]:
display(
    df_raw.select(
        "DatGeracaoConjuntoDados",
        "SigAgente",
        "NumCNPJ",
        "IdeConjUndConsumidoras",
        "DscConjUndConsumidoras",
        "SigIndicador",
        "AnoIndice",
        "NumPeriodoIndice",
        "VlrIndiceEnviado"
    ).limit(20)
)

DatGeracaoConjuntoDados,SigAgente,NumCNPJ,IdeConjUndConsumidoras,DscConjUndConsumidoras,SigIndicador,AnoIndice,NumPeriodoIndice,VlrIndiceEnviado
2026-08-06,EAC,4065033000170,12598,Marechal Thaumaturgo,FECIPC,2020,12,0.0
2026-08-06,EAC,4065033000170,12589,Santa Rosa,FECINE,2020,9,0.0
2026-08-06,CEA,5965546000109,14566,SANTA RITA,DECXNC,2020,4,0.0
2026-08-06,CEA,5965546000109,14564,TARTARUGALZINHO,DEC,2020,2,17.52
2026-08-06,ETO,25086034000171,13658,Pedro Afonso,NumCon,2020,12,13835.0
2026-08-06,ETO,25086034000171,13609,Aliança,DEC,2020,8,7.55
2026-08-06,ETO,25086034000171,13648,Nova Olinda,FECXPC,2020,10,0.0
2026-08-06,ETO,25086034000171,13646,Natividade,DECXNC,2020,8,0.0
2026-08-06,ETO,25086034000171,13625,Colinas Tocantins,NumCon,2020,5,13380.0
2026-08-06,ETO,25086034000171,16029,ARRAIAS II,DECIP,2020,8,0.02


In [0]:
from pyspark.sql import functions as F

indicator_profile = (
    df_raw
    .groupBy("SigIndicador")
    .agg(
        F.count("*").alias("row_count"),
        F.min("AnoIndice").alias("min_year"),
        F.max("AnoIndice").alias("max_year"),
        F.min("VlrIndiceEnviado").alias("min_value"),
        F.max("VlrIndiceEnviado").alias("max_value")
    )
    .orderBy(F.desc("row_count"))
)

display(indicator_profile)

SigIndicador,row_count,min_year,max_year,min_value,max_value
NumCon,244031,2020,2026,0.0,159709.0
FECXP,244031,2020,2026,0.0,4.08
FECIND,243901,2020,2026,0.0,17.55
DECXN,243901,2020,2026,0.0,56.66
DECIND,243901,2020,2026,0.0,70.51
DECXP,243901,2020,2026,0.0,9.78
FECINE,243901,2020,2026,0.0,12.55
DEC,243901,2020,2026,0.0,70.51
DECIP,243901,2020,2026,0.0,22.69
FECIPC,243901,2020,2026,0.0,2.24


In [0]:
temporal_profile = (
    df_raw
    .groupBy("AnoIndice")
    .agg(
        F.count("*").alias("row_count"),
        F.min("NumPeriodoIndice").alias("min_period"),
        F.max("NumPeriodoIndice").alias("max_period"),
        F.countDistinct("NumPeriodoIndice").alias("distinct_periods")
    )
    .orderBy("AnoIndice")
)

display(temporal_profile)

AnoIndice,row_count,min_period,max_period,distinct_periods
2020,863665,1,12,12
2021,861072,1,12,12
2022,734681,1,12,12
2023,734831,1,12,12
2024,738197,1,12,12
2025,742317,1,12,12
2026,368099,1,7,7


In [0]:
null_profile = df_raw.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
])

display(null_profile)

DatGeracaoConjuntoDados,IdeConjUndConsumidoras,DscConjUndConsumidoras,SigAgente,NumCNPJ,SigIndicador,AnoIndice,NumPeriodoIndice,VlrIndiceEnviado
0,0,0,0,0,0,0,0,0


## Grain and Business Key Analysis

The objective of this section is to determine the actual grain of the
source dataset and identify a candidate business key before designing
downstream tables.

In [0]:
candidate_key = [
    "IdeConjUndConsumidoras",
    "SigIndicador",
    "AnoIndice",
    "NumPeriodoIndice"
]

key_profile = (
    df_raw
    .groupBy(*candidate_key)
    .count()
)

duplicate_keys = (
    key_profile
    .filter(F.col("count") > 1)
)

print(f"Total rows: {df_raw.count():,}")
print(f"Distinct candidate keys: {key_profile.count():,}")
print(f"Duplicate candidate keys: {duplicate_keys.count():,}")

Total rows: 5,042,862
Distinct candidate keys: 5,042,602
Duplicate candidate keys: 260


In [0]:
consumer_set_consistency = (
    df_raw
    .groupBy("IdeConjUndConsumidoras")
    .agg(
        F.countDistinct("SigAgente").alias("agent_count"),
        F.countDistinct("NumCNPJ").alias("cnpj_count"),
        F.countDistinct("DscConjUndConsumidoras").alias("description_count")
    )
)

display(
    consumer_set_consistency
    .filter(
        (F.col("agent_count") > 1) |
        (F.col("cnpj_count") > 1) |
        (F.col("description_count") > 1)
    )
)

IdeConjUndConsumidoras,agent_count,cnpj_count,description_count


In [0]:
entity_profile = df_raw.agg(
    F.countDistinct("SigAgente").alias("distributors"),
    F.countDistinct("NumCNPJ").alias("cnpjs"),
    F.countDistinct("IdeConjUndConsumidoras").alias("consumer_sets"),
    F.countDistinct("SigIndicador").alias("indicators")
)

display(entity_profile)

distributors,cnpjs,consumer_sets,indicators
105,105,3980,23


## Duplicate Candidate Key Investigation

The initial candidate business key is not fully unique.

The objective of this analysis is to determine whether the duplicate keys
represent:

- exact duplicate records;
- different dataset generations;
- legitimate multiple observations;
- or another source-data behavior.

No records will be removed until the duplication pattern is understood.

In [0]:
duplicate_records = (
    df_raw
    .join(
        duplicate_keys.select(*candidate_key),
        on=candidate_key,
        how="inner"
    )
    .orderBy(
        "IdeConjUndConsumidoras",
        "SigIndicador",
        "AnoIndice",
        "NumPeriodoIndice",
        "DatGeracaoConjuntoDados"
    )
)

display(duplicate_records)

IdeConjUndConsumidoras,SigIndicador,AnoIndice,NumPeriodoIndice,DatGeracaoConjuntoDados,DscConjUndConsumidoras,SigAgente,NumCNPJ,VlrIndiceEnviado
13449,FECXP,2025,6,2026-08-06,AGUA VERMELHA,ELEKTRO,2328280000197,0.0
13449,FECXP,2025,6,2026-08-06,AGUA VERMELHA,ELEKTRO,2328280000197,0.0
13449,NumCon,2025,6,2026-08-06,AGUA VERMELHA,ELEKTRO,2328280000197,5094.0
13449,NumCon,2025,6,2026-08-06,AGUA VERMELHA,ELEKTRO,2328280000197,5094.0
13450,FECXP,2025,6,2026-08-06,AGUAI,ELEKTRO,2328280000197,0.0
13450,FECXP,2025,6,2026-08-06,AGUAI,ELEKTRO,2328280000197,0.0
13450,NumCon,2025,6,2026-08-06,AGUAI,ELEKTRO,2328280000197,15478.0
13450,NumCon,2025,6,2026-08-06,AGUAI,ELEKTRO,2328280000197,15478.0
13451,FECXP,2025,6,2026-08-06,AMERICO DE CAMPOS,ELEKTRO,2328280000197,0.0
13451,FECXP,2025,6,2026-08-06,AMERICO DE CAMPOS,ELEKTRO,2328280000197,0.0


In [0]:
duplicate_analysis = (
    duplicate_records
    .groupBy(*candidate_key)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("DatGeracaoConjuntoDados").alias("generation_dates"),
        F.countDistinct("VlrIndiceEnviado").alias("distinct_values"),
        F.min("DatGeracaoConjuntoDados").alias("first_generation_date"),
        F.max("DatGeracaoConjuntoDados").alias("last_generation_date")
    )
)

display(
    duplicate_analysis
    .groupBy(
        "row_count",
        "generation_dates",
        "distinct_values"
    )
    .count()
    .orderBy(
        "row_count",
        "generation_dates",
        "distinct_values"
    )
)

row_count,generation_dates,distinct_values,count
2,1,1,260


In [0]:
candidate_key_v2 = [
    "IdeConjUndConsumidoras",
    "SigIndicador",
    "AnoIndice",
    "NumPeriodoIndice",
    "DatGeracaoConjuntoDados"
]

key_profile_v2 = (
    df_raw
    .groupBy(*candidate_key_v2)
    .count()
)

duplicate_keys_v2 = (
    key_profile_v2
    .filter(F.col("count") > 1)
)

print(f"Total rows: {df_raw.count():,}")
print(f"Distinct candidate keys v2: {key_profile_v2.count():,}")
print(f"Duplicate candidate keys v2: {duplicate_keys_v2.count():,}")

Total rows: 5,042,862
Distinct candidate keys v2: 5,042,602
Duplicate candidate keys v2: 260


In [0]:
exact_duplicates = (
    df_raw
    .groupBy(*df_raw.columns)
    .count()
    .filter(F.col("count") > 1)
)

print(f"Exact duplicate groups: {exact_duplicates.count():,}")

display(
    exact_duplicates
    .groupBy("count")
    .agg(F.count("*").alias("groups"))
    .orderBy("count")
)

Exact duplicate groups: 260


count,groups
2,260


In [0]:
prototype_indicators = (
    df_raw
    .filter(F.col("SigIndicador").isin("DEC", "FEC"))
    .groupBy(
        "SigIndicador",
        "AnoIndice"
    )
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("IdeConjUndConsumidoras").alias("consumer_sets"),
        F.min("NumPeriodoIndice").alias("min_period"),
        F.max("NumPeriodoIndice").alias("max_period")
    )
    .orderBy(
        "SigIndicador",
        "AnoIndice"
    )
)

display(prototype_indicators)

SigIndicador,AnoIndice,rows,consumer_sets,min_period,max_period
DEC,2020,37559,3131,1,12
DEC,2021,37442,3131,1,12
DEC,2022,37363,3114,1,12
DEC,2023,37421,3119,1,12
DEC,2024,37584,3132,1,12
DEC,2025,37795,3150,1,12
DEC,2026,18737,3177,1,7
FEC,2020,37559,3131,1,12
FEC,2021,37442,3131,1,12
FEC,2022,37363,3114,1,12
